# Predictions of an LLM
We first just take a simple model and load into memory, and then see what the output for the next token is.

We calculate the entropy, as well as observe the distribution for the top 10 tokens.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt")

torch.manual_seed(42)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :]
probs = torch.softmax(logits, dim=-1)
entropy = -(probs * torch.log2(probs)).sum()
print(entropy.item())

topk = torch.topk(probs, 10)

for p, idx in zip(topk.values[0], topk.indices[0]):
    token = tokenizer.decode([idx.item()])
    prob = p.item()

    print(f"{repr(token):20} {prob:.4f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

4.5
' Paris'             0.2949
' ______'            0.1230
':\n'                0.0659
':\n\n'              0.0547
' located'           0.0513
' __'                0.0483
' ____'              0.0400
' the'               0.0376
' ('                 0.0259
' ['                 0.0228


# Growing certainty of predictions over time
We want to now give the LLM a small piece of text, and ask it to predict each next token in turn.

We are curious to see how the entropy for each token changes over time.

We give it a nonsense piece of text (written by me), and a piece of text reciting a famous piece of history.
Logically we expect the nonsense to be much more information dense, due to the random nature of the phrase, than the logical recitation of a well known piece of history.

In [2]:
sekigahara = """The battle of Sekigahara was fought in the plain of Sekigahara, nestled in the mountains east of Lake Biwa in central Japan.
Largely regarded as the definitive battle in the Sengoku Jidai, it was fought between supporters of Toyotomi
Hideyori, the son of Toyotomi Hideyoshi, and Tokugawa Ieyasu. The battle was a decisive victory for the
Tokugawa forces, leading to the unification of Japan and the establishment of the Tokugawa Shogunate."""

nonsense = """Blue cats are red. Following this cereal rained from the earth. Gained was everything.
Though longing for custard, television was discarded. How scissors take pens to create pepper. Totyota
are mighty ducks. Apple Jesus ran under Jupiter. Cards eaten ooze flowers within the sky. Shoes drank your
second ninja tape. Conflagration of fluffy diamonds sapped eye forks. Gratuitous toes experience Michelin temples."""

In [3]:
def predictions_over_time(text, model, tokenizer):
    tokens = tokenizer(text, return_tensors="pt")["input_ids"][0]

    torch.manual_seed(42)
    total_entropy = 0
    with torch.inference_mode():
        for i in range(len(tokens) - 1):
            # Context seen so far
            context = tokens[:i+1].unsqueeze(0)

            outputs = model(context)
            logits = outputs.logits[0, -1]
            probs = torch.softmax(logits, dim = 0)

            actual_next = tokens[i+1]
            actual_prob = probs[actual_next].item()
            predicted = probs.argmax().item()
            entropy = -(probs * torch.log2(probs)).sum().item()
            total_entropy += entropy

            print("="*40)
            print("Context   : ", repr(tokenizer.decode(context[0])))
            print("Actual    :", repr(tokenizer.decode([actual_next])))
            print("Predicted :", repr(tokenizer.decode([predicted])))
            print("P(actual) :", actual_prob)
            print("Entropy   :", entropy)
            print("Total Etropy :", total_entropy)


In [4]:
predictions_over_time(sekigahara, model, tokenizer)

Context   :  'The'
Actual    : ' battle'
Predicted : ' following'
P(actual) : 1.519918441772461e-05
Entropy   : 7.59375
Total Etropy : 7.59375
Context   :  'The battle'
Actual    : ' of'
Predicted : ' of'
P(actual) : 0.30078125
Entropy   : 5.0
Total Etropy : 12.59375
Context   :  'The battle of'
Actual    : ' Sek'
Predicted : ' the'
P(actual) : 0.000652313232421875
Entropy   : 8.75
Total Etropy : 21.34375
Context   :  'The battle of Sek'
Actual    : 'ig'
Predicted : 'ond'
P(actual) : 0.03662109375
Entropy   : 6.9375
Total Etropy : 28.28125
Context   :  'The battle of Sekig'
Actual    : 'ah'
Predicted : 'ah'
P(actual) : 0.77734375
Entropy   : 1.46875
Total Etropy : 29.75
Context   :  'The battle of Sekigah'
Actual    : 'ara'
Predicted : 'ara'
P(actual) : 0.98828125
Entropy   : 0.1259765625
Total Etropy : 29.8759765625
Context   :  'The battle of Sekigahara'
Actual    : ' was'
Predicted : ' was'
P(actual) : 0.26171875
Entropy   : 4.65625
Total Etropy : 34.5322265625
Context   :  'The bat

KeyboardInterrupt: 

In [ ]:
predictions_over_time(nonsense, model, tokenizer)

# Get probability dictionary for tokens at each step
We want to get a probability dictionary across all the tokens for each step, to hand to our huffman encoder. That we wrote separately.

In [5]:
def token_prob_dict_from_logits(logits):
    probs = torch.softmax(logits, dim=-1).squeeze()
    return {i: float(p) for i, p in enumerate(probs.tolist())}

# Will this approach work?
Before we go any further I'll just give a small proof to show that generating a new huffman code with every dictionary should work.

Say we have read our huffman code up to binary digit $b_n$. We want to read the next code word, but we don't know how many digits it is.
We hand our current reconstructed text to the large language model, which will produce a token probability dictionary in a deterministic way (since we set the seed). Since our huffman code generator is deterministic, we recover exactly the huffman code used to encode our codewords. After this point, we read digit by digit, checking if it's a valid codeword in the Huffman code.

Since the Huffman coding is unique, we only fail to find our codeword, $b_{n+1}b_{n+2}...b_{n+m}$ say, if we find some other
valid codeword while checking a smaller substring of the actual codeword.

That is to say if there exists a codeword of length $k$, with $k<m$ such that $b_{n+1}b_{n+2}...b_{k}$ is a valid codeword. But that would imply there exists a codeword that's a prefix of $b_{n+1}b_{n+2}...b{n+m}$. But this is impossible as Huffman Codes are prefix free.

So the fact that we are switching between different Huffman codes between each codeword is not a problem.

In [ ]:
import HuffmanEncoding
import numpy as np
def generate_encoding_by_different_huffman_encoding_at_each_prediciton(text, model, tokenizer):
    tokens = tokenizer(text, return_tensors="pt")["input_ids"][0]

    torch.manual_seed(42)
    expected_ideal_information = 0
    realised_ideal_information = 0
    code = ""
    total_code_length = 0
    with torch.inference_mode():
        for i in range(len(tokens) - 1):
            # Context seen so far
            context = tokens[:i+1].unsqueeze(0)

            outputs = model(context)
            logits = outputs.logits[0, -1]
            probs = torch.softmax(logits, dim = 0)

            actual_next = tokens[i+1]
            actual_prob = probs[actual_next].item()
            predicted = probs.argmax().item()
            information = -np.log2(actual_prob)
            realised_ideal_information += information
            entropy = -(probs * torch.log2(probs)).sum().item()
            expected_ideal_information += entropy

            print("="*40)
            print("Context                   : ", repr(tokenizer.decode(context[0])))
            print("Actual                    :", repr(tokenizer.decode([actual_next])))
            print("Predicted                 :", repr(tokenizer.decode([predicted])))
            print("P(actual)                 :", actual_prob)
            print("Realised token info       :", information)
            print("Model entropy             :", entropy)
            print("Cumulative token info     :", realised_ideal_information)
            print("Cumulative entropy        :", expected_ideal_information)

            token_prob_dict = token_prob_dict_from_logits(logits)
            if None in token_prob_dict:
                print("Chat GPT was wrong")
            huffman_code = HuffmanEncoding.get_huffman_code_from_frequency_dict(token_prob_dict)
            current_codeword = huffman_code[actual_next.item()]
            codeword_length = len(current_codeword)
            total_code_length += codeword_length
            code += current_codeword

            print("="*20)
            print("Token decoded             :", tokenizer.decode([actual_next.item()]))
            print("Codeword                  :", current_codeword)
            print("Codeword length           :", codeword_length)
            print("Huffman code length       :", total_code_length)
            print("Code                      :", code)
    
    return code


small_test = "A small test piece of text for testing my Huffman encoder."


generate_encoding_by_different_huffman_encoding_at_each_prediciton(small_test, model, tokenizer)
'110011110110000110000111010011010111001110100011010100111111000111011010011001000101100111100'

Context                   :  'A'
Actual                    : ' small'
Predicted                 : ' '
P(actual)                 : 0.00933837890625
Token information         : 6.742612157307348
Expected Token Information: 8.3125
Realised information      : 6.742612157307348
Expected Information      : 8.3125
Token decoded             :  small
Codeword                  : 1100111
Codeword length           : 7
Huffman code length       : 7
Code                      : 1100111
Context                   :  'A small'
Actual                    : ' test'
Predicted                 : ' town'
P(actual)                 : 0.0002574920654296875
Token information         : 11.923184402949168
Expected Token Information: 5.40625
Realised information      : 18.665796560256517
Expected Information      : 13.71875
Token decoded             :  test
Codeword                  : 101100001100
Codeword length           : 12
Huffman code length       : 19
Code                      : 1100111101100001100
Context    

'110011110110000110000111010011010111001110100011010100111111000111011010011001000101100111100'